# 03 电压门控判定与指标计算（25-GO 最终口径）

## 判定口径（本仓库最终版）
- **Voltage_Score** = 25 个电压门控 GO 标签分数的最大值。
- **Predicted = 1** ⟺ Voltage_Score ≥ 判定阈值（默认扫描 0.1–0.9）。
- 25 个 GO 标签存于 `data/labels/all_voltage_go_tags.csv`（全部为 molecular function 通道活性类，经 GO 审计确立）。

## 指标
- **Sensitivity / Specificity / MCC**：随判定阈值变化（混淆矩阵）。
- **AUROC / AUPRC**：排序指标，只依赖分数排序、与判定阈值无关。
- 5 个随机数种子（2023/1234/567/42/100）视为 5 次重复，报告 mean±SD。


In [ ]:
# -*- coding: utf-8 -*-
import csv, os, sys, math

BASE = os.path.dirname(os.getcwd())          # notebooks/ 的父目录 = 仓库根
sys.path.insert(0, BASE)

# ---------- 25-GO 判定集合 ----------
def load_volt_tags(path=None):
    """读 all_voltage_go_tags.csv -> 25 个 GO ID 集合（最终判定口径）"""
    path = path or os.path.join(BASE, "data", "labels", "all_voltage_go_tags.csv")
    tags = []
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            tags.append(r["GO_ID"])
    return tags

VOLT_TAGS = load_volt_tags()

def volt_score(gs):
    """Voltage_Score = 25 个 GO 分数最大值"""
    return max([gs.get(t, 0.0) for t in VOLT_TAGS])

# ---------- 指标 ----------
def calc_conf(y, s, thr):
    p = [1 if x >= thr else 0 for x in s]
    TP = sum(1 for a, b in zip(y, p) if a == 1 and b == 1)
    TN = sum(1 for a, b in zip(y, p) if a == 0 and b == 0)
    FP = sum(1 for a, b in zip(y, p) if a == 0 and b == 1)
    FN = sum(1 for a, b in zip(y, p) if a == 1 and b == 0)
    return TP, TN, FP, FN

def mcc_from(TP, TN, FP, FN):
    den = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    return (TP * TN - FP * FN) / den if den else 0.0

def auroc_rank(y, s):
    """平均秩法 AUROC（等价 sklearn roc_auc_score）"""
    pairs = sorted(zip(s, y), key=lambda t: t[0])
    n_pos, n_neg = sum(y), len(y) - sum(y)
    if n_pos == 0 or n_neg == 0:
        return 0.5
    ranks, i, n = [], 0, len(pairs)
    while i < n:
        j = i
        while j + 1 < n and pairs[j + 1][0] == pairs[i][0]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1
        ranks += [avg_rank] * (j - i + 1)
        i = j + 1
    sum_pos = sum(r for (_, yv), r in zip(pairs, ranks) if yv == 1)
    return (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)

def auprc_rank(y, s):
    """平均精度（等价 sklearn average_precision_score）"""
    pairs = sorted(zip(s, y), key=lambda t: -t[0])
    tp, prec_sum, n_pos = 0, 0.0, sum(y)
    if n_pos == 0:
        return 0.0
    for i, (_, yv) in enumerate(pairs, 1):
        if yv == 1:
            tp += 1
            prec_sum += tp / i
    return prec_sum / n_pos

def calc_metrics(y, s, thr):
    TP, TN, FP, FN = calc_conf(y, s, thr)
    n = len(y)
    return dict(n=n, pos=sum(y), neg=n - sum(y), TP=TP, TN=TN, FP=FP, FN=FN,
                Sensitivity=TP / (TP + FN) if TP + FN else 0.0,
                Specificity=TN / (TN + FP) if TN + FP else 0.0,
                MCC=mcc_from(TP, TN, FP, FN),
                AUROC=auroc_rank(y, s), AUPRC=auprc_rank(y, s))

def mean_sd(vals):
    m = sum(vals) / len(vals)
    sd = (sum((x - m) ** 2 for x in vals) / len(vals)) ** 0.5
    return m, sd

def wcsv(path, header, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


### 标签加载

- **5 种子 test/train**：标签来自
  - `voltage_channels_sequences_515_2023.csv`（Entry，label=1）
  - `non_voltage_channels_sequences_872_2023.csv`（Entry，label=0）
- **2026 easy**：标签来自 `labeled_sequences_for_encoding.csv`（Entry_ID + Label，812 条）。


In [ ]:
# ---------- 标签 ----------
def load_voltage_labels():
    pos = {}
    with open(os.path.join(BASE, "data", "labels", "voltage_channels_sequences_515_2023.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            pos[r["Entry"]] = 1
    neg = {}
    with open(os.path.join(BASE, "data", "labels", "non_voltage_channels_sequences_872_2023.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            neg[r["Entry"]] = 0
    lab = dict(pos); lab.update(neg)
    return lab

def load_easy_labels():
    lab = {}
    with open(os.path.join(BASE, "data", "labels", "labeled_sequences_for_encoding.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            lab[r["Entry_ID"]] = int(r["Label"])
    return lab

# 阈值扫描（0.1 - 0.9）
THRS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SEEDS = ["2023", "1234", "567", "42", "100"]
M_HDR = ["seed", "thr", "n", "pos", "neg", "TP", "TN", "FP", "FN",
         "Sensitivity", "Specificity", "MCC", "AUROC", "AUPRC"]


### 用法示例（单源扫描）

预测文件（`test2023_t010.json` 等）体积较大、未内置仓库，需先运行 01 重新生成，
或从原项目 `GO模型测试/` 复制到仓库外的 `predictions/` 目录。示例仅演示接口：


In [ ]:
# 示例：对某个预测源做阈值扫描（y=标签, s=Voltage_Score 列表）
# from _parse import load_deepgo_json   # 见 02 notebook
# gs = load_deepgo_json("../predictions/seed_2023/test2023_t010.json")
# lab = load_voltage_labels()
# y, s = [], []
# for acc in sorted(gs):
#     if acc in lab:
#         y.append(lab[acc]); s.append(volt_score(gs[acc]))
# for t in THRS:
#     print(t, calc_metrics(y, s, t))

# 直接读已交付的 25-GO 结果表
with open(os.path.join(BASE, "results", "seeds", "seed_test_25go_5seeds_metrics.csv"), encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 5:
            break
        print(row)


### 已交付结果（`results/`）
- `seeds/seed_test_25go_5seeds_metrics.csv` / `_mean_sd.csv`：5 种子 test（69 条）逐阈值 + mean±SD
- `seeds/seed_train_25go_5seeds_metrics.csv` / `_mean_sd.csv`：5 种子 train（274 条）
- `common108/train_25go_108_common_metrics*.csv`：108 条共有 entry（5 种子交叉）
- `common108/train42_anon_25go_108_common_metrics.csv`：train42 匿名预测
- `2026_easy/easy_25go_metrics_*.csv`：2026 easy 812 条（netgo3/netgo4/redundant/t010/default 五源）
